In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [2]:
DATA_DIR = Path("data/processed")

df_players = pd.read_csv(DATA_DIR / "ligue1_players_clean.csv")
df_analysis = pd.read_csv(DATA_DIR / "ligue1_team_analysis.csv")
df_form = pd.read_csv(DATA_DIR / "ligue1_team_form.csv")

print("Players:", df_players.shape)
print("Teams:", df_analysis.shape)
print("Form:", df_form.shape)

Players: (553, 180)
Teams: (18, 26)
Form: (612, 8)


In [3]:
display(df_players.head())
display(df_analysis.head())

,Joueur,Nationalité,Position,Equipe,Championnat,Age,Année_naissance,Matchs_jouées,Titularisation,Minutes,...,Actions_Défensives_Hors_Surface,Actions_Défensives_Hors_Surface/90m,Distance_Du_But_Sur_Action_Défensive,Position_Unique,Buts_per90,Assists_per90,xG_per90,G+A_per90,Buts_minus_xG,Age_Band
0,Farid El Melali,dz ALG,ATT,Angers,Ligue 1,27,1997,32,22,1893,...,NaN,NaN,NaN,ATT,0.095087,0.142631,0.171157,0.237718,-1.6,25-28
1,Josué Casimir,fr FRA,ATT.MIL,Le Havre,Ligue 1,22,2001,27,23,1997,...,NaN,NaN,NaN,ATT,0.180270,0.180270,0.148723,0.360541,0.7,21-24
2,Désiré Doué,fr FRA,ATT.MIL,Paris SG,Ligue 1,19,2005,31,18,1730,...,NaN,NaN,NaN,ATT,0.312139,0.312139,0.265318,0.624277,0.9,U21
3,Mason Greenwood,eng ENG,ATT.MIL,Marseille,Ligue 1,22,2001,34,32,2804,...,NaN,NaN,NaN,ATT,0.674037,0.160485,0.523181,0.834522,4.7,21-24
4,Zuriko Davitashvili,ge GEO,ATT.MIL,St Etienne,Ligue 1,23,2001,33,32,2778,...,NaN,NaN,NaN,ATT,0.291577,0.259179,0.262419,0.550756,0.9,21-24


,Equipe,Points,Buts_Pour,Buts_Contre,Tirs_Pour,Tirs_Contre,TirsCadrés_Pour,TirsCadrés_Contre,Matchs_Joués,xG_Pour,...,Points_Par_Match,xG_Balance,Conversion_Rate,Classement,Efficiency_Attack,Efficiency_Defense,Performance_Index,Tirs_Conversion,TirsCadrés_Conversion,Classement_Balance_xG
0,Paris SG,84,92,35,627,314,282,104,34,90.6,...,2.470588,60.2,0.146730,1,1.4,-4.6,-3.2,0.146730,0.326241,1
1,Marseille,65,74,47,476,317,205,136,34,65.5,...,1.911765,18.0,0.155462,2,8.5,0.5,9.0,0.155462,0.360976,4
2,Monaco,61,63,41,491,334,189,131,34,75.0,...,1.794118,36.7,0.128310,3,-12.0,-2.7,-14.7,0.128310,0.333333,2
3,Nice,60,66,41,489,455,195,164,34,64.4,...,1.764706,21.4,0.134969,4,1.6,2.0,3.6,0.134969,0.338462,3
4,Lille,60,52,36,435,337,166,130,34,56.6,...,1.764706,14.4,0.119540,5,-4.6,6.2,1.6,0.119540,0.313253,6


In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MIN_MINUTES_JOUEURS = 900


def prepare_players_subset(names, min_minutes=0):
    sub = df_players[df_players["Joueur"].isin(names)].copy()

    if sub.empty:
        print("Aucun joueur trouvé.")
        return None

    if "Minutes" in sub.columns:
        sub = sub[sub["Minutes"] >= min_minutes].copy()

    if sub.empty:
        print(f"Aucun joueur ne dépasse le seuil de {min_minutes} minutes.")
        return None

    if "GroupePoste" not in sub.columns:
        if "Position_Unique" in sub.columns:
            sub["GroupePoste"] = sub["Position_Unique"]
        else:
            sub["GroupePoste"] = sub["Position"]

    # Gardiens : convertir le % d'arrêts en proportion
    if "%D'arrêts" in sub.columns and "Proportion_Arrêts" not in sub.columns:
        sub["Proportion_Arrêts"] = pd.to_numeric(sub["%D'arrêts"], errors="coerce") / 100

    # Défenseurs : créer quelques variables /90 utiles si elles n'existent pas déjà
    if "Tacles" in sub.columns and "Tacles_90" not in sub.columns:
        sub["Tacles_90"] = np.where(sub["Minutes"] > 0, sub["Tacles"] * 90 / sub["Minutes"], np.nan)

    if "Interceptions" in sub.columns and "Interceptions_90" not in sub.columns:
        sub["Interceptions_90"] = np.where(sub["Minutes"] > 0, sub["Interceptions"] * 90 / sub["Minutes"], np.nan)

    if "Ballons_Bloqués" in sub.columns and "Ballons_Bloqués_90" not in sub.columns:
        sub["Ballons_Bloqués_90"] = np.where(sub["Minutes"] > 0, sub["Ballons_Bloqués"] * 90 / sub["Minutes"], np.nan)

    if "Duels_Aériens_Remportés" in sub.columns and "Duels_Aériens_Remportés_90" not in sub.columns:
        sub["Duels_Aériens_Remportés_90"] = np.where(sub["Minutes"] > 0, sub["Duels_Aériens_Remportés"] * 90 / sub["Minutes"], np.nan)

    # Milieux : créer quelques variables /90 si elles n'existent pas déjà
    if "Passes_Progressives" in sub.columns and "Passes_Progressives_90" not in sub.columns:
        sub["Passes_Progressives_90"] = np.where(sub["Minutes"] > 0, sub["Passes_Progressives"] * 90 / sub["Minutes"], np.nan)

    if "Passes_Vers_Dernier_Tiers" in sub.columns and "Passes_Vers_Dernier_Tiers_90" not in sub.columns:
        sub["Passes_Vers_Dernier_Tiers_90"] = np.where(sub["Minutes"] > 0, sub["Passes_Vers_Dernier_Tiers"] * 90 / sub["Minutes"], np.nan)

    if "Passes_Vers_Surface" in sub.columns and "Passes_Vers_Surface_90" not in sub.columns:
        sub["Passes_Vers_Surface_90"] = np.where(sub["Minutes"] > 0, sub["Passes_Vers_Surface"] * 90 / sub["Minutes"], np.nan)

    if "Portés_Vers_Lavant" in sub.columns and "Portés_Vers_LAvant_90" not in sub.columns:
        sub["Portés_Vers_LAvant_90"] = np.where(sub["Minutes"] > 0, sub["Portés_Vers_Lavant"] * 90 / sub["Minutes"], np.nan)

    if "Portés_Vers_Dernier_Tiers" in sub.columns and "Portés_Vers_Dernier_Tiers_90" not in sub.columns:
        sub["Portés_Vers_Dernier_Tiers_90"] = np.where(sub["Minutes"] > 0, sub["Portés_Vers_Dernier_Tiers"] * 90 / sub["Minutes"], np.nan)

    return sub


def plot_role_bar_comparison(sub, metrics, title):
    metrics = [m for m in metrics if m in sub.columns]

    if len(metrics) == 0:
        print("Aucune métrique disponible pour le graphique.")
        return

    data = sub[["Joueur"] + metrics].copy()
    data = data.dropna(subset=metrics, how="all")

    if data.empty or len(data) < 2:
        print("Pas assez de joueurs comparables pour afficher le graphique.")
        return

    data = data.set_index("Joueur")[metrics].T

    data.plot(
        kind="bar",
        figsize=(10, 6),
        edgecolor="black"
    )

    plt.title(title, fontsize=13, weight="bold")
    plt.ylabel("Valeur")
    plt.xlabel("")
    plt.grid(axis="y", linestyle=":", alpha=0.4)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def compare_players_by_role(names, min_minutes=0):
    sub = prepare_players_subset(names, min_minutes=min_minutes)
    if sub is None:
        return

    groups = sub["GroupePoste"].dropna().unique()

    if len(groups) != 1 or groups[0] == "AUTRE":
        print("Comparaison impossible : les joueurs n'ont pas le même type de poste (ATT / MIL / DEF / G).")
        cols_error = [c for c in ["Joueur", "Equipe", "Position", "Position_Unique", "GroupePoste", "Minutes"] if c in sub.columns]
        display(sub[cols_error])
        return

    group = groups[0]
    print(f"Comparaison pour le groupe de poste : {group}")

    role_config = {
        "ATT": {
            "table_cols": [
                "Joueur", "Equipe", "Position", "Minutes",
                "Buts", "xG", "Buts_minus_xG",
                "Buts_per90", "xG_per90", "Assists_per90", "G+A_per90"
            ],
            "plot_metrics": ["Buts_per90", "xG_per90", "Assists_per90", "G+A_per90"],
            "title": "Comparaison attaquants — production offensive"
        },
        "MIL": {
            "table_cols": [
                "Joueur", "Equipe", "Position", "Minutes",
                "Actions_Menant_A_Un_Tir/90m",
                "Passes_Progressives_90",
                "Passes_Vers_Dernier_Tiers_90",
                "Portés_Vers_LAvant_90",
                "Portés_Vers_Dernier_Tiers_90"
            ],
            "plot_metrics": [
                "Actions_Menant_A_Un_Tir/90m",
                "Passes_Progressives_90",
                "Passes_Vers_Dernier_Tiers_90",
                "Portés_Vers_LAvant_90"
            ],
            "title": "Comparaison milieux — création et progression"
        },
        "DEF": {
            "table_cols": [
                "Joueur", "Equipe", "Position", "Minutes",
                "Tacles_90",
                "Interceptions_90",
                "Ballons_Bloqués_90",
                "Duels_Aériens_Remportés_90",
                "%Duels_Aériens_Remportés"
            ],
            "plot_metrics": [
                "Tacles_90",
                "Interceptions_90",
                "Ballons_Bloqués_90",
                "Duels_Aériens_Remportés_90"
            ],
            "title": "Comparaison défenseurs — activité défensive"
        },
        "G": {
            "table_cols": [
                "Joueur", "Equipe", "Position", "Minutes",
                "Buts_Concédés/90min",
                "%D'arrêts",
                "Proportion_Arrêts",
                "PostShot_xG+/-/90m"
            ],
            "plot_metrics": [
                "Buts_Concédés/90min",
                "Proportion_Arrêts",
                "PostShot_xG+/-/90m"
            ],
            "title": "Comparaison gardiens — performance"
        }
    }

    if group not in role_config:
        print("Poste non géré.")
        cols_error = [c for c in ["Joueur", "Equipe", "Position", "GroupePoste", "Minutes"] if c in sub.columns]
        display(sub[cols_error])
        return

    config = role_config[group]

    table_cols = [c for c in config["table_cols"] if c in sub.columns]
    if table_cols:
        display(sub[table_cols].round(2))

    plot_role_bar_comparison(sub, config["plot_metrics"], config["title"])

In [59]:
import ipywidgets as widgets
from IPython.display import display


def compare_players_widget(min_minutes=MIN_MINUTES_JOUEURS):
    players_all = sorted(df_players["Joueur"].dropna().unique())
    selected_players = []

    search = widgets.Text(
        value="",
        placeholder="Rechercher un joueur...",
        description="🔍",
        layout=widgets.Layout(width="300px"),
    )

    dropdown = widgets.Dropdown(
        options=players_all,
        description="Résultats :",
        layout=widgets.Layout(width="400px"),
    )

    add_btn = widgets.Button(
        description="Ajouter",
        button_style="success",
        layout=widgets.Layout(width="120px"),
    )

    clear_btn = widgets.Button(
        description="Réinitialiser",
        button_style="warning",
        layout=widgets.Layout(width="120px"),
    )

    compare_btn = widgets.Button(
        description="Comparer",
        button_style="primary",
        layout=widgets.Layout(width="120px"),
    )

    selected_label = widgets.HTML()
    out = widgets.Output()

    def update_dropdown(filter_text=""):
        ft = str(filter_text).lower().strip()
        filtered = [p for p in players_all if ft in p.lower()]

        if filtered:
            dropdown.options = filtered
            dropdown.value = filtered[0]
        else:
            dropdown.options = []
            dropdown.value = None

    def refresh_selected_label():
        if selected_players:
            selected_label.value = "<b>Joueurs sélectionnés :</b> " + ", ".join(selected_players)
        else:
            selected_label.value = "<b>Joueurs sélectionnés :</b> (aucun)"

    def on_search_change(change):
        update_dropdown(change["new"])

    def on_add_clicked(_):
        if dropdown.value is None:
            return
        if dropdown.value not in selected_players:
            selected_players.append(dropdown.value)
        refresh_selected_label()

    def on_clear_clicked(_):
        selected_players.clear()
        refresh_selected_label()
        with out:
            out.clear_output()

    def on_compare_clicked(_):
        with out:
            out.clear_output()
            if not selected_players:
                print("Sélectionne au moins un joueur.")
                return

            names = sorted(selected_players)
            print(f"Joueurs retenus : {names}")
            compare_players_by_role(names, min_minutes=min_minutes)

    search.observe(on_search_change, names="value")
    add_btn.on_click(on_add_clicked)
    clear_btn.on_click(on_clear_clicked)
    compare_btn.on_click(on_compare_clicked)

    update_dropdown("")
    refresh_selected_label()

    ui = widgets.VBox([
        search,
        dropdown,
        widgets.HBox([add_btn, clear_btn, compare_btn]),
        selected_label,
        out
    ])

    display(ui)

In [45]:
compare_players_widget()

In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def build_team_table():
    """
    Construit une table propre pour la comparaison d'équipes
    à partir de df_analysis.
    """
    cols = [
        "Equipe",
        "Classement",
        "Points",
        "Buts_Pour",
        "Buts_Contre",
        "Diff_Buts",
        "xG_Pour",
        "PostShot_xG_Subi",
        "xG_Balance",
        "Conversion_Rate"
    ]

    cols = [c for c in cols if c in df_analysis.columns]
    return df_analysis[cols].copy()


teams_table = build_team_table()


def prepare_teams_subset(team_names):
    sub = teams_table[teams_table["Equipe"].isin(team_names)].copy()

    if sub.empty:
        print("Aucune équipe trouvée.")
        return None

    if "Classement" in sub.columns:
        sub = sub.sort_values("Classement")
    elif "Points" in sub.columns:
        sub = sub.sort_values("Points", ascending=False)

    return sub


def plot_team_bar_comparison(sub, metrics, title):
    metrics = [m for m in metrics if m in sub.columns]

    if len(metrics) == 0:
        print("Aucune métrique disponible pour le graphique.")
        return

    data = sub[["Equipe"] + metrics].copy()
    data = data.dropna(subset=metrics, how="all")

    if data.empty or len(data) < 2:
        print("Pas assez d'équipes comparables pour afficher le graphique.")
        return

    data = data.set_index("Equipe")[metrics].T

    data.plot(
        kind="bar",
        figsize=(10, 6),
        edgecolor="black"
    )

    plt.title(title, fontsize=13, weight="bold")
    plt.ylabel("Valeur")
    plt.xlabel("")
    plt.grid(axis="y", linestyle=":", alpha=0.4)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def compare_teams(team_names):
    sub = prepare_teams_subset(team_names)
    if sub is None:
        return

    print("Comparaison des équipes sélectionnées")

    table_cols = [
        "Equipe",
        "Classement",
        "Points",
        "Buts_Pour",
        "Buts_Contre",
        "Diff_Buts",
        "xG_Pour",
        "PostShot_xG_Subi",
        "xG_Balance",
        "Conversion_Rate"
    ]
    table_cols = [c for c in table_cols if c in sub.columns]
    display(sub[table_cols].round(2))

    if len(sub) < 2:
        print("Sélectionne au moins deux équipes pour afficher les graphiques.")
        return

    # Graphique 1 : performance générale
    plot_team_bar_comparison(
        sub,
        ["Points", "Diff_Buts", "xG_Balance"],
        "Comparaison équipes — performance générale"
    )

    # Graphique 2 : profil offensif / défensif
    plot_team_bar_comparison(
        sub,
        ["Buts_Pour", "Buts_Contre", "xG_Pour", "PostShot_xG_Subi"],
        "Comparaison équipes — attaque et défense"
    )

In [63]:
import ipywidgets as widgets
from IPython.display import display


def compare_teams_widget():
    teams_all = sorted(teams_table["Equipe"].dropna().unique())
    selected_teams = []

    search = widgets.Text(
        value="",
        placeholder="Rechercher une équipe...",
        description="🔍",
        layout=widgets.Layout(width="300px")
    )

    dropdown = widgets.Dropdown(
        options=teams_all,
        description="Résultats :",
        layout=widgets.Layout(width="400px")
    )

    add_btn = widgets.Button(
        description="Ajouter",
        button_style="success",
        layout=widgets.Layout(width="120px")
    )

    clear_btn = widgets.Button(
        description="Réinitialiser",
        button_style="warning",
        layout=widgets.Layout(width="120px")
    )

    compare_btn = widgets.Button(
        description="Comparer",
        button_style="primary",
        layout=widgets.Layout(width="120px")
    )

    selected_label = widgets.HTML()
    out = widgets.Output()

    def update_dropdown(filter_text=""):
        ft = str(filter_text).lower().strip()
        filtered = [t for t in teams_all if ft in t.lower()]

        if filtered:
            dropdown.options = filtered
            dropdown.value = filtered[0]
        else:
            dropdown.options = []
            dropdown.value = None

    def refresh_selected_label():
        if selected_teams:
            selected_label.value = "<b>Équipes sélectionnées :</b> " + ", ".join(selected_teams)
        else:
            selected_label.value = "<b>Équipes sélectionnées :</b> (aucune)"

    def on_search_change(change):
        update_dropdown(change["new"])

    def on_add_clicked(_):
        if dropdown.value is None:
            return
        if dropdown.value not in selected_teams:
            selected_teams.append(dropdown.value)
        refresh_selected_label()

    def on_clear_clicked(_):
        selected_teams.clear()
        refresh_selected_label()
        with out:
            out.clear_output()

    def on_compare_clicked(_):
        with out:
            out.clear_output()
            if not selected_teams:
                print("Sélectionne au moins une équipe.")
                return

            print(f"Équipes retenues : {selected_teams}")
            compare_teams(selected_teams)

    search.observe(on_search_change, names="value")
    add_btn.on_click(on_add_clicked)
    clear_btn.on_click(on_clear_clicked)
    compare_btn.on_click(on_compare_clicked)

    update_dropdown("")
    refresh_selected_label()

    ui = widgets.VBox([
        search,
        dropdown,
        widgets.HBox([add_btn, clear_btn, compare_btn]),
        selected_label,
        out
    ])

    display(ui)

In [65]:
compare_teams_widget()